# Notebook 01 — Data Exploration (EDA)
**EPPS–American Airlines Data Challenge — GROW 26.2**

**Goal:** Understand both inbound and outbound tables before any modeling.

Outputs saved to `../outputs/figures/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

INBOUND_PATH  = '../data/raw/inbound_training_table_enriched.csv'
OUTBOUND_PATH = '../data/raw/outbound_training_table_enriched.csv'
FIGURES_PATH  = '../outputs/figures/'
os.makedirs(FIGURES_PATH, exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('✅ Imports OK')

## 1 — Load Data

In [ ]:
inbound  = pd.read_csv(INBOUND_PATH)
outbound = pd.read_csv(OUTBOUND_PATH)

# Drop fully-null rows in outbound (data artifact)
outbound = outbound.dropna(subset=['airport_B']).reset_index(drop=True)

# Parse dates
inbound['Date']  = pd.to_datetime(inbound['Date'])
outbound['Date'] = pd.to_datetime(outbound['Date'])

print('=== INBOUND ===')
print(f'  Rows    : {len(inbound):,}')
print(f'  Columns : {len(inbound.columns)}')
print(f'  Dates   : {inbound["Date"].min().date()} → {inbound["Date"].max().date()}')
print()
print('=== OUTBOUND ===')
print(f'  Rows    : {len(outbound):,}')
print(f'  Columns : {len(outbound.columns)}')
print(f'  Dates   : {outbound["Date"].min().date()} → {outbound["Date"].max().date()}')

## 2 — Column Overview

In [ ]:
print('INBOUND COLUMNS:')
for i,c in enumerate(inbound.columns,1): print(f'  {i:2}. {c}')
print()
print('OUTBOUND COLUMNS:')
for i,c in enumerate(outbound.columns,1): print(f'  {i:2}. {c}')
print()
only_in  = set(inbound.columns)  - set(outbound.columns)
only_out = set(outbound.columns) - set(inbound.columns)
print('Only in INBOUND :',  sorted(only_in))
print('Only in OUTBOUND:', sorted(only_out))

## 3 — Null Analysis

In [ ]:
def null_report(df, name):
    n = df.isnull().sum()
    n = n[n > 0]
    if len(n) == 0:
        print(f'{name}: ✅ No nulls')
        return
    pct = (n / len(df) * 100).round(1)
    print(f'{name} nulls:')
    print(pd.DataFrame({'count': n, '%': pct}).to_string())
    print()

null_report(inbound,  'INBOUND')
null_report(outbound, 'OUTBOUND')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for ax, df, title in zip(axes, [inbound, outbound], ['Inbound Nulls', 'Outbound Nulls']):
    np_data = (df.isnull().mean() * 100).sort_values(ascending=False)
    np_data = np_data[np_data > 0]
    if len(np_data) == 0:
        ax.text(0.5, 0.5, 'No nulls ✅', ha='center', va='center', fontsize=14)
    else:
        np_data.plot(kind='barh', ax=ax, color='salmon', edgecolor='white')
        ax.axvline(50, color='red', ls='--', alpha=0.5, label='50%')
        ax.legend()
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Null %')
plt.suptitle('Null Value Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}01_null_distribution.png', bbox_inches='tight')
plt.show()
print('💾 01_null_distribution.png')

## 4 — Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, df, title in zip(axes, [inbound, outbound],
                         ['Inbound — A→DFW', 'Outbound — DFW→B']):
    counts = df['delayed'].value_counts().reindex([0,1]).fillna(0)
    pcts   = df['delayed'].value_counts(normalize=True).reindex([0,1]).fillna(0) * 100
    bars   = ax.bar(['Not Delayed (0)', 'Delayed (1)'], counts.values,
                    color=['#2ecc71','#e74c3c'], edgecolor='white', width=0.5)
    for bar, cnt, pct in zip(bars, counts.values, pcts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(counts)*0.02,
                f'{int(cnt):,}\n({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    ax.set_ylim(0, max(counts)*1.25)
plt.suptitle('Target Variable — Class Balance Check', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}02_target_distribution.png', bbox_inches='tight')
plt.show()
print('💾 02_target_distribution.png')

## 5 — Delay Rate by Season

In [ ]:
season_order = ['winter','spring','summer','fall']
season_colors = {'winter':'#3498db','spring':'#2ecc71','summer':'#e74c3c','fall':'#f39c12'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, title in zip(axes, [inbound, outbound],
                         ['Inbound Delay Rate by Season', 'Outbound Delay Rate by Season']):
    data = df.groupby('season')['delayed'].mean() * 100
    data = data.reindex([s for s in season_order if s in data.index])
    cols = [season_colors.get(s,'gray') for s in data.index]
    bars = ax.bar(data.index, data.values, color=cols, edgecolor='white', width=0.5)
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
    ax.axhline(df['delayed'].mean()*100, color='black', ls='--', alpha=0.5, label='Overall avg')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Delay Rate (%)')
    ax.set_ylim(0, 100)
    ax.legend()
plt.suptitle('Seasonal Delay Patterns', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}03_delay_by_season.png', bbox_inches='tight')
plt.show()
print('💾 03_delay_by_season.png')

## 6 — Delay Rate by Hour of Day

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, df, hcol, title in zip(
    axes, [inbound, outbound],
    ['dep_hour_A','dep_hour_DFW'],
    ['Inbound: Delay Rate by Dep Hour (Airport A)',
     'Outbound: Delay Rate by Dep Hour (DFW)']):
    hr_delay = df.groupby(hcol)['delayed'].mean() * 100
    hr_count = df.groupby(hcol)['delayed'].count()
    ax2 = ax.twinx()
    ax2.bar(hr_count.index, hr_count.values, alpha=0.2, color='steelblue')
    ax2.set_ylabel('Flight Count', color='steelblue')
    ax.plot(hr_delay.index, hr_delay.values, 'o-', color='#e74c3c', lw=2.5, ms=6)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Hour (UTC)')
    ax.set_ylabel('Delay Rate (%)', color='#e74c3c')
    ax.set_ylim(0, 100)
plt.suptitle('Delay Rate by Hour of Day', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}04_delay_by_hour.png', bbox_inches='tight')
plt.show()
print('💾 04_delay_by_hour.png')

## 7 — Delay Rate by Weekday

In [ ]:
day_map = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, title in zip(axes, [inbound, outbound],
                         ['Inbound Delay by Weekday','Outbound Delay by Weekday']):
    data = df.groupby('weekday')['delayed'].mean() * 100
    data.index = [day_map.get(int(d), d) for d in data.index]
    colors = ['#e74c3c' if d in ['Sat','Sun'] else '#3498db' for d in data.index]
    bars = ax.bar(data.index, data.values, color=colors, edgecolor='white', width=0.6)
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{val:.0f}%', ha='center', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Delay Rate (%)')
    ax.set_ylim(0, 100)
    ax.axhline(df['delayed'].mean()*100, color='k', ls='--', alpha=0.4)
plt.suptitle('Delay Rate by Weekday (Red = Weekend)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}05_delay_by_weekday.png', bbox_inches='tight')
plt.show()
print('💾 05_delay_by_weekday.png')

## 8 — Delay Column Distributions

In [ ]:
delay_cols = ['gate_dep_delay','apt_dep_delay','gate_arr_delay',
              'block_delay','airborne_delay','taxi_out_delay','taxi_in_delay']
avail = [c for c in delay_cols if c in inbound.columns]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for i, col in enumerate(avail):
    ax = axes[i]
    inbound[inbound['delayed']==0][col].hist(ax=ax, bins=25, alpha=0.65,
        color='#2ecc71', label='Not Delayed', density=True)
    inbound[inbound['delayed']==1][col].hist(ax=ax, bins=25, alpha=0.65,
        color='#e74c3c', label='Delayed', density=True)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_xlabel('Minutes')
    ax.legend(fontsize=7)
for j in range(len(avail), len(axes)): axes[j].set_visible(False)
plt.suptitle('Inbound Delay Columns — Delayed vs Not Delayed', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}06_delay_distributions.png', bbox_inches='tight')
plt.show()
print('💾 06_delay_distributions.png')

## 9 — Weather Feature Distributions

In [ ]:
wx_cont = ['dfw_temp_f','dfw_humidity_pct','dfw_wind_kts','dfw_visibility_mi',
           'origin_temp_f','origin_humidity_pct','origin_wind_kts','origin_visibility_mi']
avail_wx = [c for c in wx_cont if c in inbound.columns]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for i, col in enumerate(avail_wx):
    ax = axes[i]
    inbound[inbound['delayed']==0][col].dropna().hist(
        ax=ax, bins=25, alpha=0.65, color='#2ecc71', label='Not Delayed', density=True)
    inbound[inbound['delayed']==1][col].dropna().hist(
        ax=ax, bins=25, alpha=0.65, color='#e74c3c', label='Delayed', density=True)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)
for j in range(len(avail_wx), len(axes)): axes[j].set_visible(False)
plt.suptitle('Weather Features — Delayed vs Not Delayed', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}07_weather_distributions.png', bbox_inches='tight')
plt.show()
print('💾 07_weather_distributions.png')

## 10 — Binary Weather vs Delay Rate

In [ ]:
bin_wx = ['dfw_thunderstorm','dfw_precipitation','dfw_freezing','dfw_low_ceiling',
          'origin_thunderstorm','origin_precipitation','origin_freezing','origin_low_ceiling']
avail_b = [c for c in bin_wx if c in inbound.columns]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for i, col in enumerate(avail_b):
    ax = axes[i]
    pivot = inbound.groupby(col)['delayed'].mean() * 100
    colors_b = ['#2ecc71','#e74c3c']
    bars = ax.bar(pivot.index.astype(str), pivot.values,
                  color=colors_b[:len(pivot)], edgecolor='white', width=0.5)
    for bar, val in zip(bars, pivot.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
    ax.set_xticks(range(len(pivot)))
    ax.set_xticklabels(['No','Yes'][:len(pivot)])
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_ylabel('Delay %')
    ax.set_ylim(0, 100)
for j in range(len(avail_b), len(axes)): axes[j].set_visible(False)
plt.suptitle('Binary Weather Feature vs Delay Rate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}08_binary_weather_delay.png', bbox_inches='tight')
plt.show()
print('💾 08_binary_weather_delay.png')

## 11 — Correlation with Delay

In [ ]:
num_cols = inbound.select_dtypes(include=[np.number]).columns.tolist()
corr = inbound[num_cols].corr()['delayed'].drop('delayed')
top20 = corr.abs().sort_values(ascending=False).head(20)
corr_plot = corr[top20.index]

fig, ax = plt.subplots(figsize=(10, 7))
colors_c = ['#e74c3c' if v>0 else '#3498db' for v in corr_plot.values]
ax.barh(corr_plot.index, corr_plot.values, color=colors_c, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Top 20 Features Correlated with Delay (Inbound)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Correlation coefficient')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}09_correlation_delay.png', bbox_inches='tight')
plt.show()
print('💾 09_correlation_delay.png')
print()
print('Top 10 positive:')
print(corr_plot[corr_plot>0].sort_values(ascending=False).head(10).to_string())
print()
print('Top 5 negative:')
print(corr_plot[corr_plot<0].sort_values().head(5).to_string())

## 12 — Airport Coverage

In [ ]:
print(f'Unique Airport A (inbound origins) : {inbound["airport_A"].nunique()}')
print(f'Unique Airport B (outbound dests)  : {outbound["airport_B"].nunique()}')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, df, apcol, title in zip(
    axes, [inbound, outbound], ['airport_A','airport_B'],
    ['Top 20 Airport A (Inbound Origins)','Top 20 Airport B (Outbound Dests)']):
    top = (df.groupby(apcol).agg(cnt=('delayed','count'), dr=('delayed','mean'))
             .sort_values('cnt', ascending=False).head(20))
    cmap_vals = plt.cm.RdYlGn_r(top['dr'].values)
    bars = ax.barh(top.index, top['cnt'], color=cmap_vals, edgecolor='white')
    for bar, (_, row) in zip(bars, top.iterrows()):
        ax.text(bar.get_width()+max(top['cnt'])*0.01,
                bar.get_y()+bar.get_height()/2,
                f'{row["dr"]*100:.0f}%', va='center', fontsize=8, color='#c0392b')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Flight Count')
plt.suptitle('Airport Coverage — Bar = Flight Count, Red % = Delay Rate',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}10_airport_coverage.png', bbox_inches='tight')
plt.show()
print('💾 10_airport_coverage.png')

## 13 — Monthly Trend

In [ ]:
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, df, title in zip(axes, [inbound, outbound],
                         ['Inbound Monthly Delay Rate','Outbound Monthly Delay Rate']):
    monthly = df.groupby('month').agg(dr=('delayed','mean'), cnt=('delayed','count'))
    monthly.index = [month_map.get(int(m), m) for m in monthly.index]
    ax2 = ax.twinx()
    ax2.bar(monthly.index, monthly['cnt'], alpha=0.2, color='steelblue')
    ax2.set_ylabel('Flight Count', color='steelblue')
    ax.plot(monthly.index, monthly['dr']*100, 'o-', color='#e74c3c', lw=2.5, ms=7)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Delay Rate (%)', color='#e74c3c')
    ax.set_ylim(0, 100)
plt.suptitle('Monthly Delay Trend (2025)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}11_monthly_trend.png', bbox_inches='tight')
plt.show()
print('💾 11_monthly_trend.png')

## 14 — EDA Summary

In [ ]:
print('='*55)
print('  EDA SUMMARY')
print('='*55)
print(f'\n📊 Dataset')
print(f'  Inbound rows  : {len(inbound):,}')
print(f'  Outbound rows : {len(outbound):,}')
in_dr  = inbound['delayed'].mean()*100
out_dr = outbound['delayed'].mean()*100
print(f'\n🎯 Class Balance')
print(f'  Inbound  delay rate : {in_dr:.1f}%')
print(f'  Outbound delay rate : {out_dr:.1f}%')
print(f'  → SMOTE needed: {"Yes" if min(in_dr,out_dr)<30 else "No"}')
print(f'\n✈️  Airports')
print(f'  Unique A : {inbound["airport_A"].nunique()}')
print(f'  Unique B : {outbound["airport_B"].nunique()}')
print(f'\n✅ Next → Notebook 02: Data Cleaning')
print('='*55)